In [8]:
%pip install pandas openpyxl
%pip install dash plotly pandas

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 24.4 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [3]:
# Load the file
df = pd.read_excel('1740CAF.xlsx')

# Data Cleaning & Preparation
# Turning 'Grant Amount' column into numeric values
df['Grant Amount'] = pd.to_numeric(df['Grant Amount'].replace(r'[\$,]', '', regex=True), errors='coerce')
df['Grant Amount'] = df['Grant Amount'].fillna(0)
df['Grant Amount'] = df['Grant Amount'].astype(int).round(2)

# Turn Project Category into a list of unique categories
df['Project Category'] = df['Project Category'].str.strip()
df['Project Category'] = df['Project Category'].str.split(r',\s*')
df_exploded = df.explode('Project Category')
df_grouped = df_exploded.groupby(['Year', 'Project Category'])['Grant Amount'].sum().reset_index()
categories = df_grouped['Project Category'].unique()
# print(categories)

# Show the first 5 rows
df.head()

,Year,Date,Grant Amount,Project,Project Category,Organization/Applicant,Status,Description,Additional Notes,Location/Municipality
0,2015,2015-03-04 00:00:00,4000,support Carnegie Science Center Future Cities ...,[Education/Competition],Carnegie Science Center future cities program,NaN,to support the Carnegie Science Center Future ...,NaN,North Shore City; Pitt
1,2015,2015-05-06 00:00:00,50000,planting trees,[Tree Planting],Tree Pittsburgh,NaN,to plant trees on industrial properties in La...,NaN,"Lawrenceville, Chateau, and Neville Island"
2,2015,2015-05-06 00:00:00,225000,discourage open burning and educate on the hea...,[Education/Competition],Outreach Campaign,NaN,To fund an outreach campaign in response to th...,NaN,NaN
3,2015,2015-07-08 00:00:00,337600,Northgate Asthma Initiative,[Public Health/Medicine],Healthy Homes Program,NaN,To support the Northgate Asthma Initiative / c...,NaN,NaN
4,2015,2015-09-02 00:00:00,400000,purchase of solar panels to light parking lots...,"[Emission Reduction, Diesel Reduction/Electrif...",Solar Panels,NaN,for the purchase of solar panels to light park...,2 Categories,Allegheny County


In [4]:
# Spending by Year

# Data Cleaning & Preparation
# Group by Year and sum the Grant Amount
df_yearly = df.groupby('Year')['Grant Amount'].sum().reset_index()
# Check the transformed data
print(df_yearly.head())

# Create the Line Chart
fig = px.line(df_yearly, 
            x='Year', 
            y='Grant Amount', 
            title='Annual Spending Trend (2015-2024)',
            markers=True, # Adds dots to the line for easier hovering
            template='plotly_white')

# Enhance the Interactivity & Look
fig.update_traces(line_color="#4d5cfd", line_width=3) # Custom line color
fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Spending ($)",
    hovermode="x unified", # Shows all data for that year in one tooltip
    yaxis=dict(
        title="Grant Amount",
        tickprefix="$",    # Adds $ to the start of every number
        tickformat=",",    # Adds commas (e.g., 1,000,000)
    )
)

fig.show()

# Export for Wix
# html_snippet = fig.to_html(full_html=False, include_plotlyjs='cdn')
# print(html_snippet)

   Year  Grant Amount
0  2015       1016600
1  2016       1579146
2  2017         72500
3  2018       2154962
4  2019       1326865


In [5]:
# Spending by Project Category

# --- CREATE THE FIGURE ---
fig = go.Figure()

# Add a line for each category
for cat in categories:
    temp_df = df_grouped[df_grouped['Project Category'] == cat]
    fig.add_trace(
        go.Scatter(
            x=temp_df['Year'], 
            y=temp_df['Grant Amount'], 
            name=cat, 
            mode='lines+markers', # Added markers for better visibility
            visible=True
        )
    )

# --- CREATE THE DROPDOWN & LAYOUT ---
fig.update_layout(
    updatemenus=[
        {
            "buttons": [
                {
                    "label": "Show All",
                    "method": "update",
                    "args": [{"visible": [True] * len(categories)}, {"title": "All Project Spending"}]
                },
                *[
                    {
                        "label": cat,
                        "method": "update",
                        "args": [
                            {"visible": [c == cat for c in categories]},
                            {"title": f"Spending: {cat}"}
                        ]
                    } for cat in categories
                ]
            ],
            "direction": "down",
            "showactive": True,
            "x": 0.0, # Moved to far left
            "y": 1.2  # Moved slightly higher to avoid overlapping the title
        }
    ],
    width=1000,
    height=600,
    autosize=False,
    margin=dict(l=50, r=50, t=100, b=50),
    xaxis=dict(title="Year", tickmode='linear'), # Ensures every year shows up
    yaxis=dict(
        title="Grant Amount",
        tickprefix="$",    # Adds $ to the start of every number
        tickformat=",",    # Adds commas (e.g., 1,000,000)
    ),
    title={
            'text': "Project Spending",
            'y': 0.95,           # Vertical position (0 to 1)
            'x': 0.5,            # Horizontal position (0.5 is center)
            'xanchor': 'center', # Ensures the center of the text aligns with the x coordinate
            'yanchor': 'top'
    }
)

fig.show()

# Export for Wix
html_snippet = fig.to_html(full_html=False, include_plotlyjs='cdn')
print(html_snippet)

<div>                        <script type="text/javascript">window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
        <script charset="utf-8" src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>                <div id="80e3a448-171b-4d4b-b2ca-f3a1b8392ce2" class="plotly-graph-div" style="height:600px; width:1000px;"></div>            <script type="text/javascript">                                    window.PLOTLYENV=window.PLOTLYENV || {};                                    if (document.getElementById("80e3a448-171b-4d4b-b2ca-f3a1b8392ce2")) {                    Plotly.newPlot(                        "80e3a448-171b-4d4b-b2ca-f3a1b8392ce2",                        [{"mode":"lines+markers","name":"Diesel Reduction\u002fElectrification","visible":true,"x":[2015,2016,2020,2023,2024],"y":[400000,24000,25000,2198339,1105061],"type":"scatter"},{"mode":"lines+markers","name":"Education\u002fCompetition","visible":true,"x":[2015,2016,2017,2020,2023],"y":[229000,82000,72500,131495,9000

In [119]:
# Hannah


In [18]:
# Abby
import pandas as pd
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Load Excel file
df = pd.read_csv('1740CAF(DataSet1).csv', encoding='latin-1')

# Clean Grant Amount
df['Grant Amount'] = (
    df['Grant Amount']
    .astype(str)
    .str.replace(r'[\$,\s]', '', regex=True)
    .astype(float)
)

# Clean category column
df['Project Category 1'] = df['Project Category 1'].str.strip()

# Aggregate data
cat_spend = (
    df.groupby('Project Category 1')['Grant Amount']
    .sum()
    .reset_index()
    .sort_values('Grant Amount', ascending=False)
)

# --- PIE CHART ---
fig = px.pie(
    cat_spend,
    names='Project Category 1',
    values='Grant Amount',
    title='Grant Spending Distribution by Category',
    hole=0.4  # donut style
)

fig.update_traces(
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>$%{value:,.0f}<br>%{percent}'
)

fig.update_layout(height=500)

fig.show()

In [19]:
import sys
!{sys.executable} -m pip install plotly

In [21]:
# Neha

import pandas as pd
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

# Load data
df = pd.read_csv('1740CAF(DataSet1).csv', encoding='latin-1')
#df = pd.read_excel("CAF_DS.xlsx", sheet_name="DataSet1")

# Clean grant amount column
df["Grant Amount"] = df["Grant Amount"].replace(r'[\$,]', '', regex=True).astype(float)

# Replace N/A with NaN
df["Project Category 2"] = df["Project Category 2"].replace("N/A", pd.NA)

# Get list of all categories
categories = pd.unique(
    pd.concat([df["Project Category 1"], df["Project Category 2"]]).dropna()
)

app = Dash(__name__)

app.layout = html.Div([
    
    html.H2("Grant Funding by Year"),

    dcc.Checklist(
        id='category_selector',
        options=[{"label": c, "value": c} for c in categories] +
                [{"label": "All Categories", "value": "ALL"}],
        value=["ALL"],
        inline=False
    ),

    dcc.Graph(id='grant_graph')
])


@app.callback(
    Output("grant_graph", "figure"),
    Input("category_selector", "value")
)

def update_graph(selected_categories):

    if "ALL" in selected_categories:
        filtered = df.copy()
    else:
        filtered = df[
            df["Project Category 1"].isin(selected_categories) |
            df["Project Category 2"].isin(selected_categories)
        ]

    # Remove duplicate rows so grant amount isn't counted twice
    filtered = filtered.drop_duplicates()

    # Sum grant money by year
    yearly = filtered.groupby("Year")["Grant Amount"].sum().reset_index()

    fig = px.line(
        yearly,
        x="Year",
        y="Grant Amount",
        markers=True,
        title="Grant Money Allocated Per Year"
    )

    return fig


if __name__ == "__main__":
    app.run(debug=True)

In [ ]:
# Lasya
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/Users/lasyanedunuri/Downloads/1740CAF(DataSet1) (1).csv', encoding='latin-1') 

# Clean Grant Amount: remove $, commas, spaces → float
df['Grant Amount'] = (
    df['Grant Amount']
    .astype(str)
    .str.replace(r'[\$,\s]', '', regex=True)
    .astype(float)
)

# Normalize category columns (strip whitespace)
df['Project Category 1'] = df['Project Category 1'].str.strip()
df['Project Category 2'] = df['Project Category 2'].str.strip()

print(f"Dataset shape: {df.shape}")
print(f"Years covered: {sorted(df['Year'].unique())}")
print(f"Total grant funding: ${df['Grant Amount'].sum():,.0f}")
df.head(10)

cat_spend = (
    df.groupby('Project Category 1')['Grant Amount']
    .sum()
    .reset_index()
    .sort_values('Grant Amount', ascending=False)
)

fig = px.bar(
    cat_spend,
    x='Project Category 1',
    y='Grant Amount',
    title='Total Grant Spending by Project Category',
    color='Grant Amount',
    color_continuous_scale='Teal',
    text='Grant Amount'
)
fig.update_traces(
    texttemplate='$%{text:,.0f}',
    textposition='outside'
)
fig.update_layout(
    xaxis_title='Project Category',
    yaxis_title='Total Grant Amount ($)',
    xaxis_tickangle=-30,
    coloraxis_showscale=False,
    height=500
)
fig.show()

Dataset shape: (69, 4)
Years covered: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Total grant funding: $17,412,879


In [ ]:
# Filter for Emission Reduction only
emission_df = df[
    (df['Project Category 1'] == 'Emission Reduction') |
    (df['Project Category 2'] == 'Emission Reduction')
]

# Group by Year
emission_yearly = (
    emission_df.groupby('Year')['Grant Amount']
    .sum()
    .reset_index()
)

# Line chart over time
fig = px.line(
    emission_yearly,
    x='Year',
    y='Grant Amount',
    title='Emission Reduction Grant Spending Over Time',
    markers=True,
    template='plotly_white'
)
fig.update_traces(
    line_color="#2ca02c",
    line_width=3,
    hovertemplate="<b>Year:</b> %{x}<br><b>Spent:</b> $%{y:,.0f}<extra></extra>"
)
fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Grant Amount ($)",
    yaxis_tickprefix="$",
    yaxis_tickformat=",",
    hovermode="x unified"
)
fig.show()

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import SelectMultiple, VBox, Output
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

categories = sorted(df['Project Category 1'].dropna().unique().tolist())

# Dropdown widget (hold Ctrl/Cmd to select multiple)
selector = SelectMultiple(
    options=categories,
    value=[categories[0]],
    description='Category:',
    layout={'width': '300px', 'height': '200px'}
)

out = Output()

def update_chart(change):
    selected = list(selector.value)
    out.clear_output(wait=True)

    # For each row, assign it to whichever selected category is its Project Category 1
    # If Cat 1 not selected but Cat 2 is, still label it by Cat 1
    mask = (
        df['Project Category 1'].isin(selected) |
        df['Project Category 2'].isin(selected)
    )
    filtered = df[mask].copy()

    # Label each row by its Cat 1 if selected, otherwise Cat 1 anyway (per your rule)
    filtered['Display Category'] = filtered['Project Category 1']

    yearly = (
        filtered.groupby(['Year', 'Display Category'])['Grant Amount']
        .sum()
        .reset_index()
    )

    fig = px.line(
        yearly,
        x='Year',
        y='Grant Amount',
        color='Display Category',
        markers=True,
        template='plotly_white',
        title='Grant Spending Over Time by Category'
    )
    fig.update_traces(
        line_width=3,
        hovertemplate="<b>Year:</b> %{x}<br><b>Spent:</b> $%{y:,.0f}<extra></extra>"
    )
    fig.update_layout(
        xaxis_title="Year",
        yaxis_title="Total Grant Amount ($)",
        yaxis_tickprefix="$",
        yaxis_tickformat=",",
        hovermode="x unified"
    )

    with out:
        fig.show()

selector.observe(update_chart, names='value')
update_chart(None)  # render on load

display(VBox([selector, out]))